# Задания 1 и 2: Случайный лес на данных Adult Dataset

На основе блокнота из лабораторной работы 5 (Задание 4)

In [1]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

## Загрузка и подготовка данных

In [2]:
# Загрузка данных
column_names = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 
                'marital-status', 'occupation', 'relationship', 'race', 'sex', 
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']

data_train = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
    names=column_names,
    na_values=' ?',
    skipinitialspace=True
)

data_test = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test',
    names=column_names,
    na_values=' ?',
    skipinitialspace=True,
    skiprows=1
)

print(f"Train shape: {data_train.shape}")
print(f"Test shape: {data_test.shape}")

Train shape: (32561, 15)
Test shape: (16281, 15)


In [3]:
# Удаление пропусков
data_train = data_train.dropna()
data_test = data_test.dropna()

# Очистка целевой переменной
data_test['income'] = data_test['income'].str.rstrip('.')

print(f"После удаления пропусков:")
print(f"Train: {data_train.shape}, Test: {data_test.shape}")

После удаления пропусков:
Train: (32561, 15), Test: (16281, 15)


In [4]:
# Разделение на признаки и целевую переменную
X_train = data_train.drop('income', axis=1)
y_train = data_train['income'].apply(lambda x: 1 if x == '>50K' else 0)

X_test = data_test.drop('income', axis=1)
y_test = data_test['income'].apply(lambda x: 1 if x == '>50K' else 0)

print(f"Распределение классов в обучающей выборке:")
print(y_train.value_counts())

Распределение классов в обучающей выборке:
income
0    24720
1     7841
Name: count, dtype: int64


In [5]:
# Кодирование категориальных признаков
categorical_columns = X_train.select_dtypes(include=['object']).columns

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

for col in categorical_columns:
    le = LabelEncoder()
    X_train_encoded[col] = le.fit_transform(X_train[col])
    X_test_encoded[col] = le.transform(X_test[col])

print(f"Данные закодированы. Число признаков: {X_train_encoded.shape[1]}")

Данные закодированы. Число признаков: 14


## Базовая модель: Дерево решений (из лабораторной 5)

In [6]:
# Обучение дерева решений
tree = DecisionTreeClassifier(max_depth=9, random_state=17)
tree.fit(X_train_encoded, y_train)

tree_predictions = tree.predict(X_test_encoded)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print(f"Дерево решений - Accuracy: {tree_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, tree_predictions))

Дерево решений - Accuracy: 0.8511

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.95      0.91     12435
           1       0.76      0.54      0.63      3846

    accuracy                           0.85     16281
   macro avg       0.82      0.74      0.77     16281
weighted avg       0.84      0.85      0.84     16281



---
## Задание 1: Случайный лес без настройки параметров

In [7]:
# Обучение случайного леса с параметрами по умолчанию
rf_default = RandomForestClassifier(n_estimators=100, random_state=17)
rf_default.fit(X_train_encoded, y_train)

# Прогноз на тестовой выборке
rf_default_predictions = rf_default.predict(X_test_encoded)

# Accuracy
rf_default_accuracy = accuracy_score(y_test, rf_default_predictions)
print(f"Случайный лес (без настройки) - Accuracy: {rf_default_accuracy:.4f}")

# Отчет классификации
print("\nClassification Report:")
print(classification_report(y_test, rf_default_predictions))

Случайный лес (без настройки) - Accuracy: 0.8542

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91     12435
           1       0.73      0.61      0.67      3846

    accuracy                           0.85     16281
   macro avg       0.81      0.77      0.79     16281
weighted avg       0.85      0.85      0.85     16281



In [8]:
# Сравнение с деревом решений
print("="*60)
print("СРАВНЕНИЕ: Дерево решений vs Случайный лес (без настройки)")
print("="*60)
print(f"Дерево решений:          {tree_accuracy:.4f}")
print(f"Случайный лес:           {rf_default_accuracy:.4f}")
print(f"Улучшение:               {(rf_default_accuracy - tree_accuracy):.4f} ({(rf_default_accuracy - tree_accuracy)*100:.2f}%)")
print("="*60)

СРАВНЕНИЕ: Дерево решений vs Случайный лес (без настройки)
Дерево решений:          0.8511
Случайный лес:           0.8542
Улучшение:               0.0031 (0.31%)


---
## Задание 2: Случайный лес с настройкой параметров

In [ ]:
# Параметры для GridSearchCV
forest_params = {
    'max_depth': range(10, 21),
    'max_features': range(5, 105, 10)
}

# GridSearchCV
locally_best_forest = GridSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=17, n_jobs=-1),
    forest_params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

# Обучение
locally_best_forest.fit(X_train_encoded, y_train)

# Лучшие параметры
print("Best params:", locally_best_forest.best_params_)
print("Best cross validation score:", locally_best_forest.best_score_)

Fitting 5 folds for each of 110 candidates, totalling 550 fits


In [ ]:
# Прогноз на тестовой выборке
tuned_forest_predictions = locally_best_forest.predict(X_test_encoded)

# Accuracy
tuned_forest_accuracy = accuracy_score(y_test, tuned_forest_predictions)
print(f"Случайный лес (настроенный) - Accuracy: {tuned_forest_accuracy:.4f}")

# Отчет классификации
print("\nClassification Report:")
print(classification_report(y_test, tuned_forest_predictions))

---
## Итоговое сравнение всех моделей

In [ ]:
# Сравнительная таблица
results = pd.DataFrame({
    'Модель': [
        'Дерево решений',
        'Случайный лес (без настройки)',
        'Случайный лес (настроенный)'
    ],
    'Accuracy': [
        tree_accuracy,
        rf_default_accuracy,
        tuned_forest_accuracy
    ],
    'Улучшение vs Дерево': [
        0,
        rf_default_accuracy - tree_accuracy,
        tuned_forest_accuracy - tree_accuracy
    ]
})

results['Улучшение (%)'] = results['Улучшение vs Дерево'] * 100
results = results.sort_values('Accuracy', ascending=False)

print("\n" + "="*80)
print("ИТОГОВОЕ СРАВНЕНИЕ МОДЕЛЕЙ")
print("="*80)
print(results.to_string(index=False))
print("="*80)

## Выводы

1. **Случайный лес без настройки** показал улучшение по сравнению с деревом решений за счет усреднения предсказаний множества деревьев
2. **Настройка гиперпараметров** (max_depth, max_features) дополнительно улучшила результаты
3. Ансамблевые методы снижают переобучение и повышают обобщающую способность модели